In [1]:
# === Imports ===
import pickle
import numpy as np  # Array operations
import matplotlib.pyplot as plt  # Plotting
import pandas as pd  # Data handling
import pyp2options  # Custom library --> Calibrations
import astropy.units as u  # Physical units
import scipy.optimize as spopt  # Optimization algorithms
import sys

from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from scipy.stats import binned_statistic, median_abs_deviation
from scipy.interpolate import UnivariateSpline, LSQUnivariateSpline
from scipy.interpolate import interp1d

## rcfit Code Snippet

In [14]:
# === Defined Parameters ===
field_name = 'MISHAPS_F1_DITH0'  # Field identifier
chip_id = 'N10'  # Chip identifier
magnitude_band = 'r'  # Band used for magnitude measurement
color_band_1 = 'r'  # First band for color (e.g. r-z)
color_band_2 = 'z'  # Second band for color
field_chip_label = f"{field_name}_{chip_id}"  # Combined label for file references

# CMD (Color-Magnitude Diagram) Parameters
default_color_cut = 1.06
magnitude_cut = 17.3
color_range_min = 0.0
color_range_max = 2.0
msto_shift_value = 3.5  # Main Sequence Turn-Off shift

# Adjust defaults for specific color combinations
if color_band_2 == "z" and color_band_1 == "r":
    default_color_cut = 0.6 # OG default_color_cut = 0.46
    color_range_max = 1.0
    msto_shift_value = 3.0
    
# Enable interactive plots if enough command-line arguments are provided
interactive_mode = False
if len(sys.argv) >= 6:
    interactive_mode = True

# Optionally override color and magnitude cuts
if len(sys.argv) >= 7:
    default_color_cut = float(sys.argv[6])
    print("Color cut set to:", default_color_cut)

if len(sys.argv) >= 8:
    magnitude_cut = float(sys.argv[7])
    print("Magnitude cut set to:", magnitude_cut)

print("Using magnitude and color bands:", magnitude_band, color_band_1, color_band_2)

# Optional positional filtering via RA/Dec limits
ra_min, ra_max = -1, -1
dec_min, dec_max = -1, -1
apply_position_cut = False

if len(sys.argv) >= 9:
    ra_min = float(sys.argv[8])
    ra_max = float(sys.argv[9])
    dec_min = float(sys.argv[10])
    dec_max = float(sys.argv[11])
    apply_position_cut = True

Using magnitude and color bands: r r z


## Defining Functions

In [2]:
def gmi_diam_zang(color, i, d):
    '''
    Calculate stellar radii from g-i and i, based on Zang 2018
    https://ui.adsabs.harvard.edu/abs/2018PASP..130j4401Z/abstract
    ---
    color : (g-i) color for the stars
    i : i-band magnitude for the stars
    d : distance to the stars in kpc
    '''
    # log_theta = c0 + c1(X-Xp)
    X = color
    log_diam_0 = 0.920 + 0.316*(X-0.576)
    log_diam_g = log_diam_0 - 0.2*i
    stellar_diams = 10**log_diam_g # in mas
    
    # r = diam/2*w
    r_sol = 0.00465 # AU/R_sol
    w = 1/d # parallax, [mas]
    rads = stellar_diams/(2*w) # AU
    rads_sol = rads/r_sol # AU/(AU/R_sol) = AU*R_sol/AU = R_sol
    
    return stellar_diams, rads_sol

In [3]:
# === Isochrone Filter and Radii Calculation Function ===

'''
This function takes the name of your isochrone file and a log(age), filters the isochrone file for the appropriate 
    age, and calculates the stellar radii from the luminosity, temperature, etc.

The age of the bulge is another complicated question, but we've been using 10^10 years as the age. 
    So log(age) should just be 10.
'''
def load_iso(isofile,logages_):
    # these are just column numbers
    cRadius=0
    cMass=3
    cMini=2
    cMI=13
    cMTESS=-2
    cMGaia=-5
    cLum=6
    cTeff=4
    cAge=1
    logages = logages_
    
    # if you want multiple ages, give them as a list and this will keep all of them. Otherwise just give a single one.
    if isinstance(logages_,list):
        logages = np.array(logages_)
    elif not isinstance(logages_,np.ndarray):
        logages = np.array([logages])
    
    # Load isochrone, skipping rows that just have comments
    iso = pd.read_csv(isofile, sep='\\s+', header=None, skiprows=13, engine='python', comment='#')
                                                                                            
    #Replace the idx column with the radii                                                                  
    iso.iloc[:,0] = 10**(0.5*iso.iloc[:,cLum]-2*(iso.iloc[:,cTeff]-np.log10(5777.0)))
            
    # this does the actual filtering for the age(s) you want
    ret = []
    for i, logage in enumerate(logages):
        ret.append(iso[(iso.iloc[:,-1] <= 4.0) & (np.abs(iso.iloc[:,cAge]-logage)<0.001)])
    if len(ret)==1:
        ret = ret[0]
    
    return ret

## Loading in Files and Naming Columns + Filters

In [4]:
# === Load in Pickle Files ===
pipedir = '/home/bernthold.17/SURP2025/'
rmz_gmi_spline_file = open(f"{pipedir}/Transforms/psrmz_psgmi_transform.pickle", 'rb')
rmz_imz_spline_file = open(f"{pipedir}/Transforms/psrmz_psimz_transform.pickle", 'rb')

rmz_gmi_spline = pickle.load(rmz_gmi_spline_file)
rmz_imz_spline = pickle.load(rmz_imz_spline_file)

rmz_gmi_spline_file.close()
rmz_imz_spline_file.close()

/tmp/slurmtmp.8842042/ipykernel_1068162/1832519988.py:6: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  rmz_gmi_spline = pickle.load(rmz_gmi_spline_file)
/tmp/slurmtmp.8842042/ipykernel_1068162/1832519988.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to acce

In [5]:
# === Load in PanSTARRS Isochrone File ===
isochrone_file = "/home/bernthold.17/SURP2025/Transforms/MIST_v1.2_feh_p0.00_afe_p0.0_vvcrit0.4_PanSTARRS.iso.cmd"
print(f"Reading isochrone file from: {isochrone_file}")

logage = 10
isochrone = load_iso(isochrone_file, logage)

isochrone_file_columns = ['Radius_Estimate', 'log10_isochrone_age_yr', 'initial_mass', 'star_mass', 'log_Teff', 'log_g', 'log_L', '[Fe/H]_init',
                          '[Fe/H]', 'PS_g', 'PS_r', 'PS_i', 'PS_z', 'PS_y', 'PS_w', 'PS_open', 'phase']
isochrone.columns = isochrone_file_columns

Reading isochrone file from: /home/bernthold.17/SURP2025/Transforms/MIST_v1.2_feh_p0.00_afe_p0.0_vvcrit0.4_PanSTARRS.iso.cmd


/tmp/slurmtmp.8842042/ipykernel_1068162/1633689941.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0         1.449754
1         1.470619
2         1.510297
3         1.550594
4         1.591545
            ...   
105589    0.015550
105590    0.015499
105591    0.015449
105592    0.015401
105593    0.015353
Length: 105594, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  iso.iloc[:,0] = 10**(0.5*iso.iloc[:,cLum]-2*(iso.iloc[:,cTeff]-np.log10(5777.0)))


In [6]:
# === Load Configuration from Options File ===
options = pyp2options.pip2options('/home/bernthold.17/SURP2025/fit_models/')

/home/bernthold.17/SURP2025/Transforms/pyp2options.py:12: ParserWarning: Skipping line 65: expected 2 fields, saw 5
Skipping line 68: expected 2 fields, saw 5
Skipping line 71: expected 2 fields, saw 5
Skipping line 74: expected 2 fields, saw 5
Skipping line 80: expected 2 fields, saw 5
Skipping line 83: expected 2 fields, saw 5

  df = pd.read_csv(pipedir + '/pip2options.sh',sep='=',header=None,comment="#",on_bad_lines=on_bad_lines)


In [9]:
# Column indices for quality flags
qflag_mag = options['cmdq-' + magnitude_band] - 1
qflag_c1 = options['cmdq-' + color_band_1] - 1
qflag_c2 = options['cmdq-' + color_band_2] - 1

# Column indices for magnitudes
mag_col_index = options['cmdm-' + magnitude_band] - 1
c1_col_index = options['cmdm-' + color_band_1] - 1
c2_col_index = options['cmdm-' + color_band_2] - 1

# Column indices for position
ra_col_index = 1
dec_col_index = 2

In [10]:
# Reference directory for outputs
reference_directory = '/home/bernthold.17/SURP2025/fit_models/'

In [11]:
# === Load and Filter Photometric Catalog ===
catalog_path = "/home/bernthold.17/SURP2025/Data_Files/" + field_chip_label + ".cmd.calibrate_ps"
print(f"Reading catalog from: {catalog_path}")

Reading catalog from: /home/bernthold.17/SURP2025/Data_Files/MISHAPS_F1_DITH0_N10.cmd.calibrate_ps


In [12]:
# Load the photometric catalog with mixed data types
try:
    photometric_catalog_df = pd.read_csv(catalog_path, delim_whitespace=True, comment='#', header=None)
    photometric_catalog = photometric_catalog_df.values  # convert to NumPy array for compatibility
except Exception as error:
    print(f"Error loading catalog: {error}")
    sys.exit(1)

/tmp/slurmtmp.8842042/ipykernel_1068162/631138773.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  photometric_catalog_df = pd.read_csv(catalog_path, delim_whitespace=True, comment='#', header=None)


### Quality and Position Filters

In [15]:
print(f"Total number of stars before filters: {len(photometric_catalog)}")
# Apply magnitude quality flag filters
valid_quality = (
    #(photometric_catalog[:, qflag_mag] == 1) &
    (photometric_catalog[:, qflag_c1] == 1) &
    (photometric_catalog[:, qflag_c2] == 1)
)
filtered_catalog = photometric_catalog[valid_quality]
print(f"Stars passing quality cuts: {len(filtered_catalog)}")

# Optional position-based filtering using RA/Dec limits
if apply_position_cut:
    ra_values = filtered_catalog[:, ra_col_index]
    dec_values = filtered_catalog[:, dec_col_index]
    within_bounds = (
        (ra_values > ra_min) & (ra_values < ra_max) &
        (dec_values > dec_min) & (dec_values < dec_max)
    )
    filtered_catalog = filtered_catalog[within_bounds]

print(f"Stars within RA/Dec bounds: {len(filtered_catalog)}")

Total number of stars before filters: 101166
Stars passing quality cuts: 34630
Stars within RA/Dec bounds: 34630


### Convert from Absolute to Apparent Magnitude

In [25]:
d = 8000 # distance to the Bulge in parsecs, this will be needed for converting to apparent magnitude

ps1_iso_g = iso.loc[:,'PS_g'] + 5*np.log10(d) - 5
ps1_iso_r = iso.loc[:,'PS_r'] + 5*np.log10(d) - 5
ps1_iso_i = iso.loc[:,'PS_i'] + 5*np.log10(d) - 5
ps1_iso_z = iso.loc[:,'PS_z'] + 5*np.log10(d) - 5

### Using Extinction and Reddening Values from rcfit code to Estimate Stellar Radii

In [16]:
R_v = 3.1 # Ratio of total to selective extinction for the Milky Way, Rv = Av / E(B-V)
# fliter_lambda = [0.473, 0.642, 0.784, 0.926] # g, r, i, z, in microns from DECam

r_rc = 15.871257956510505
r_rc_err = 0.053224580334176604

A_r = 0.7953376549059392
A_r_err = 0.053224580334176604

E_rmz = 0.342295253987441
E_rmz_err = 0.0058351795630540515

In [17]:
z_rc = 15.079085656946738
z_rc_err = 0.040293617212654884

A_z = 0.00316535534217266
A_z_err = 0.040293617212654884

E_rmz_z = 0.34838788091343875
E_rmz_z_err = 0.0051848781085602706

In [24]:
r_dred = r_rc - A_r
r_dred_err = r_rc_err - A_r_err

z_dred = z_rc - A_z
z_dred_err = z_rc_err - A_z_err
print(f'Dereddening in the r-Band = {r_dred} ± {r_dred_err}, ' f'Dereddening in the z-Band= {z_dred} ± {z_dred_err}')

Dereddening in the r-Band = 15.075920301604565 ± 0.0, Dereddening in the z-Band= 15.075920301604565 ± 0.0
